## Random Forest Model
- This model got 0.86662 (top 76%). This serves as a baseline / benchmark for my other models

## XGBoost
- This model got 0.87546 (top 71%). This is a slight improvement on my random forest model

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_curve, auc

# Load the data
data = pd.read_csv('data/train.csv')
X = pd.get_dummies(data.drop(['Exited', 'CustomerId', 'Surname'], axis=1), columns=['Geography', 'Gender'])
y = data['Exited']

# Make train-dev-test split (80-10-10)
X_train, X_dev, y_train, y_dev = train_test_split(X, y, test_size=0.2)
test_data = pd.read_csv('data/test.csv')
X_test = pd.get_dummies(test_data.drop(['CustomerId', 'Surname'], axis=1), columns=['Geography', 'Gender'])

In [ ]:
# Train a Random Forest Classifier
random_forest = RandomForestClassifier()
random_forest.fit(X_train, y_train)

In [ ]:
# Make predicitons using train split
preds = random_forest.predict_proba(X_dev)[:, 1]
fpr, tpr, thresholds = roc_curve(y_dev, preds)
roc_auc = auc(fpr, tpr)
print(f"{roc_auc=}")

In [ ]:
import xgboost as xgb

# Train an XGBoost model

xgb_model = xgb.XGBClassifier(objective='binary:logistic')
xgb_model.fit(X_train, y_train)

In [ ]:
# Make predictions
preds = xgb_model.predict_proba(X_dev)[:, 1]
fpr, tpr, thresholds = roc_curve(y_dev, preds)
roc_auc = auc(fpr, tpr)
print(f"{roc_auc=}")

In [ ]:
final_xgb = xgb.XGBClassifier(objective='binary:logistic') 
final_xgb.fit(X, y)
final_xgb_preds = final_xgb.predict_proba(X_test)[:,1]

In [ ]:
# Make submission file for xgb
submission = pd.DataFrame({'id': X_test['id'],
                           'Exited': final_xgb_preds
                          })

submission.to_csv('xgboost.csv', index=False)

In [ ]:
# Train model again using all data
final_random_forest = RandomForestClassifier()
final_random_forest.fit(X, y)
final_preds = final_random_forest.predict_proba(X_test)[:, 1]

In [ ]:
# Make submission file for rf
submission = pd.DataFrame({'id': X_test['id'],
                           'Exited': final_preds
                          })

submission.to_csv('random_forest.csv', index=False)